In [1]:
print('되나? 좀 되라.. ')

되나? 좀 되라.. 


In [ ]:
## 이 노트북은 RAG를 테스트 용도입니다.
## 일반적으로 웹에서 사용하는 용도가 아닌, 프로그램을 만들어서 LLM모델을 사용하려고 한다
## LLM 모델은 : Gemini(google)-무료제공버전  참고:GPT(openai)-유료
## 컨셉 : 메뉴데이터를 벡터데이터베이스 구축 한 후 사용자 질의에 응답하는 챗봇

In [2]:
# llm 모델테스트, 
# 우리가 사용할 llm 모델은 구글 제미나이이니다.
# 그래서 구글ai studio에서 키를 발급 받아야 한다. 이 키를  api key
# 그리고 이 키를 로컬에서 관리합니다. 그래서 .env라는 폴더를 먼저 만들어요. 위치꼭 기억
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()  # .env 에 있는 구글 api 키를 자동으로 로딩

# 모델을 선정하고 파라미터 값을 설정할 수 있다.
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", #문장을 잘 만들어 주는 뇌..문장을 잘 분석하는 뇌.. 언어를 잘 이해
                             temperature=0.3,  # 0.0 일관성 높은 답변, 0.3 0.0보다는 창의성이 추가-일반적인답변
                                              #1.0 기본값- 창의적인 답변..  1.5는 정말 창의적인 답변
                                             # 창의적인 것을 llm 단어를 보고 의도를 보고 그때 그때 답변이 달라질 수있다.
                             max_output_tokens=1024 #대답에 대한 토큰 수 설정. 토큰을 판단하는 기준.
                             )
# response = llm.invoke('대한민국 소개해 줘')
# print(response.content)

# 모델이 정상 동작임을 확인했음..  RAG중 G부분에서 사용할 모델 테스트 완료 됨.

## 연구할 항목 : 모델별로 성능이 차이가 있다.. 개발 주제와 가장 적당한 모델을 찾는 것.. 
## 주제는 이렇고, 데이터는 이렇고. 그래서 open ai사의 gptx, google 제미나이 3.5X버전을 사용한 결과
## 어떤 모델의 토큰량과 정답도를 고려하여 최종 000 모델로 선택함..

In [3]:
## RAG 처럼 테스해 볼께요
query = "자장면은 달고 짜면서 맛있어.   짬뽕은 얼큰하고 맛있어.. 이게 내가 주는 데이터야.. 이 데이터로 광고문구만들어줘"
response = llm.invoke(query)
print(response.content)

## llm이 만들어 준 응답으로 서비스를 제공한다..




[{'type': 'text', 'text': '제공해주신 데이터를 바탕으로, 타겟과 채널(SNS, 배달앱, 전단지 등)에 맞춰 사용할 수 있는 **다양한 스타일의 광고 문구**를 작성해 드립니다.\n\n---\n\n### 1. 직관적이고 임팩트 있는 한 줄 카피 (배달앱 메인, 팝업용)\n> **"입안 가득 터지는 단.짠.의 중독성 \'자장면\'**  \n> **속까지 뻥 뚫리는 속 깊은 얼큰함 \'짬뽕\'**  \n> 고민하지 마세요, 둘 다 확실', 'extras': {'signature': 'EtgYCtUYAWkUfRMa0Ep9/a9n2d5yV49G4jCdFgPqeQC/8WnKjIqHNJ3Eq8com+unVgXXrb9oadPpiATeU5ygnq7yp+zkr/bFaVmXZQkMgOmqEB9s9r5J1XfAe3KrtW/RQnLJXECG+i+9vkW3Y2279FJvnjItATDfoj984eXw/E70kV2PjPmRE28iuIXNzzpYkZc36nuDxut/I/FrCaw28h3rnqCF/AQ3Q8AAc9sf+LjZhVn1m0pDtxrF2F0dx89ixCGYrRqm7+6zNXHybX162mQDgYYyWKYDu6U5j115Ly8X3iWmhtYCl5nnBa+a36DhvyylepG0eBlE332HM/dFSHkOiutSKcEiyTe7Zmd5+1VJzN9hRpGvNOicNtZ+T/C5uKD+mTzGO1OoamEK6W/Zg3bSA/CY79yXOiq1q1I1u8/OKYn2/Fn9QgUYsth0ISegKWwqVvL9T5Aomg5K252E9ygr9V0e9Cto3lZfZTv9TGxtI2XHcnx1oKw9IMSvokHB/A3JWuObQSGEH0YL5x3Rv1Mzr+Q6tYU3RF2yhss3x/I+J/Ie1QtwSfIBir6hrOZX8Yv4VOJl7FEfR+4zx06NUaST12QfDtbT4bGajugBY9V9s6op+cdLItnMfAYEpuuaeRN3/jiIvrKmPQ8aAXbXuGSsJ+bKvXZM27l0GjfTFrWizfbNZAzb/fXcyFsvkB3KGhW3ERz2gt88HW

In [5]:
## 1. LLM에게 전달 할때 우리가게(음식가게)의 메뉴의 정보를 제공해야죠
## 그러기 위해서 .. 메뉴의 정보를 불러오고 > 벡터데이터로 구축해야 한다.
## 우리정보는 어디에 있나요?  menus_recommendation.json
import json
from langchain_core.documents import Document  
#벡터디비 만들때는 데이터(우리는 json)를 Document로 만들어야함
#Document 객체는 키의 이름이 page_content와  metadata 로 되어 있음 (단순 암기 영역)

#우리의 json 데이터 로드
with open("../storage/menus_recommendation.json", "r", encoding="utf-8") as f:
    data = json.load(f)
#print(data)
#실제로는 데이터가 몇 만건일 수 있다. 이래야지. 성능이 좋게 나온다.
#일부만 확인한다. 이유: 데이터가 정확하게 로딩되었는지 확인, 한글이 깨졌는지 확인 등
print(data[:1]) # 앞에서 부터 3개
print(len(data))

#데이터를 가져왔으니.. 벡터데이터베이스를 만들기 전에 Documents 문서 객체로 변환
# Document생성
docs=[] # document 문서를 저장할 배열
for row in data:
    #content변수는 문자열입니다. 이름:자장면 설명:자장면설명 이렇게 하나의 문자열로 만든다.
    content=f"""
       이름:{row['name']} 
       설명:{row['description']}
    """
    #print(content)
    docmake = Document(
        page_content=content.strip(),
        metadata={
            "name":row['name'],
            "calories":int(row['calories']),
            "recommendation":int(row['recommendation'])
        }
    )
    docs.append(docmake)
print(docs[:2])  # 리스트에는 document 타입의 객체가 저장되어 있음

[{'id': 5, 'name': '마파두부', 'calories': 400, 'description': '두부와 다진 고기를 매콤한 소스로 볶은 중국 요리', 'recommendation': 92}]
97
[Document(metadata={'name': '마파두부', 'calories': 400, 'recommendation': 92}, page_content='이름:마파두부 \n       설명:두부와 다진 고기를 매콤한 소스로 볶은 중국 요리'), Document(metadata={'name': '깐풍기', 'calories': 750, 'recommendation': 313}, page_content='이름:깐풍기 \n       설명:튀긴 닭고기를 매콤달콤한 소스에 볶은 요리')]


In [6]:
# 위에서 생성한  documnet 가 저장된  docs리스트로 벡터 데이터 베이스를 생성해야 한다.
# 이때 docs문장에 있는 한글을 숫자화 해야 합니다.
# 가장 좋은 성능을 보이는 트랜스포머로 만든 모델을 사용할 것입니다.
# 한글을 벡터화 시키는 것을 임베딩모델이라고 하고,  우리가 사용할 임베딩 모델은
# 트랜스포머 방식으로 개발된 모델입니다.
from langchain_community.embeddings import HuggingFaceEmbeddings

#jhgan/ko-sroberta-multitask 이 임베딩 모델을 다운로드 받는 과정..
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)

C:\Users\kkk\AppData\Local\Temp\ipykernel_17728\1277726796.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
C:\Users\kkk\AppData\Local\Temp\ipykernel_17728\1277726796.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
d:\Data\260812Tr\menus_project_rag\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://

In [ ]:
## 벡터데이터베이스를 구축한다
## rag 에서 사용하는 벡터데이터베이스는 정형데이터베이스랑 다르다.
## 종류는  pincorn  faiss   chroma 디비등이 있다. 우리는 chroma db를 사용하겠다.
## 그래서 디비가 저장된 디렉토리를 생성함 .. chroma_db

## 벡터베이터베이스 생성
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,    # 벡터 데이터 베이스를 구축할 정보
    embedding=embeddings,  # docs의 문장을 해석하여 데이터베이스를 구축할 임베딩 객체
    persist_directory="../chroma_db/menus_db_meta"  # 데이터베이스 위치
)

## 2번 실행하면 덮어쓰기가 아니라 추가가 됩니다.

In [8]:
## 벡터데이터베이스 구축된 정보 확인

print("저장된 문서 개수 :", vectorstore._collection.count())

저장된 문서 개수 : 97


In [9]:
## 벡터데이터베이스를 한 번 구축하면 다음에는 구축안해도 되고 읽어 오면 됩니다.
## 이미 구축된 데이터베이스에서 데이터를 읽오 오는 법

vectorstore = Chroma(
    persist_directory="../chroma_db/menus_db_meta",
    embedding_function=embeddings  # 주의할 점.. 벡터디비구축시 embedding 모델과
                                   # 읽어 올 때의 모델은 동일해야 합니다. 
)

In [19]:
vectorstore

In [ ]:
## 지금까지의 작업으로 선정한 데이터(json)으로 벡터데이터베이스를 구축함.
## 사용자 질의로 벡터데이터베이스와 유사도가 높은 문서를 검색해보자

#사용자질문
query = "칼로리 500 이하 튀겨서 만든 요리 추천해 줘"
results = vectorstore.similarity_search(query, k=4)

for doc in results:
    print(doc.page_content)

# 치킨은 칼로리가 900 이죠.. 
# 이렇게 질의를 하니까 검색을 못한다.. 제대로 못한다..


이름:치킨 
       설명:닭고기를 바삭하게 튀기거나 구워 만든 음식
이름:깐풍기 
       설명:튀긴 닭고기를 매콤달콤한 소스에 볶은 요리
이름:돈가스 
       설명:돼지고기에 빵가루를 입혀 튀긴 음식
이름:양념치킨 
       설명:튀긴 닭고기에 매콤달콤한 양념을 입힌 음식


In [ ]:
## 파이프구성
## 질문을 합니다.. 이 질문을 llm에 분석을 시킨다.. 얻고 싶은 데이터는 질문, 칼로리, 추천수 3개의 정보를 얻어냄
## 질문과 칼로리와 추천수의 정보로 이때 유사도 검색을 함..
## 유사도 검색 후.. 증강  ... 문맥생성은 동일함..
## 중간에 질의문을 분석하도록 llm을 사용하고..
## llm이 분석한 결과(질문, 칼로리, 추천수)로 유사도 검색을 한다 > 증강 > 문맥생성(llm)

In [13]:
# 여기서 사용하는 llm의 목적은 질의문의 의도를 분석하려고 한다.
# 질의문을 분석해서 원하는질문, 칼로리, 추천수를 먼저 분석하고.. 유사도 검색에 활용..
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser # llm응답시 문자열로 해라 객체

def analyze_query(user_query):
    analyze_prompt = ChatPromptTemplate.from_template("""
    사용자의 질문을 분석하여 아래 json형식으로 결과를 출력해 줘(다른 설명은 절대 붙이지 말 것)
    - query : 벡터검색을 이한 핵심 키워드 또는 메뉴이름 (예:"매운 요리", "마라탕")
    - max_calories : 지문에 언급된 최대 칼로리 조건(숫자만 추출, 없으면 null)
    - min_recommendation : 질문에 언급된 최소 추천수 조건 (숫자만 추출, 없으면 null)

    [사용자 질문]:{query}
    [출력형식(json)]:
    {{"query":"추출된 검색어","max_calories":null, "min_recommendation":null}}
    """)

    chain = analyze_prompt | llm | StrOutputParser()
    result_str = chain.invoke({"query":user_query})  #StrOutputParser result_str 문자열

    try:
        return json.loads(result_str) ## result_str값을 json으로 변경할때 예외가 발생하면
    except Exception as e:
        return {"query":user_query,"max_calories":None, "min_recommendation":None}

#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_ana = analyze_query(userquery)
print(results_ana)
    

{'query': '매운 면 요리', 'max_calories': 1000, 'min_recommendation': None}


In [ ]:
## 메타데이터를 사용해서 유사도 검색.. -RAG에서 R단계
def retrival_docs(query, max_calories=None, min_recommendation=None, k=3):
    filter_conditions={} 
    if max_calories is not None:
        filter_conditions['calories'] = {"$lte" : int(max_calories)}
        #                  키정의      = 값.. $lte(less than or equal)  이하..   
        #                                    $lt 미만, 
        #                                    $gte(greater than or equal)  이상,   $gt   초과
    if min_recommendation is not None:
        filter_conditions['recommedation'] = {"gte" : int(min_recommendation)}

    #유사도검색시 전달한 메타데이터   filter_conditions 있다면 filter_conditions리턴, 아니면 None
    search_filter = filter_conditions if filter_conditions else None 
    print(query, search_filter)

    # 이제 유사도 검색 실시
    results = vectorstore.similarity_search(query, k=k, filter=search_filter)

    print(results)
    return results


#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_ana = analyze_query(userquery)
query = results_ana['query']
max_cal =results_ana['max_calories']
min_rec = results_ana['min_recommendation']

retrival_docs(query, max_cal, min_rec)   # 노트북에서는 마지막 코드를 출력해서 보여줘요
print('test end')
    

매운 면 요리 {'calories': {'$lte': 1000}}
[Document(id='6890858f-24da-4add-b7d1-e89c28663a82', metadata={'calories': 500, 'name': '쫄면', 'recommendation': 226}, page_content='이름:쫄면 \n       설명:쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='c66a2cba-c5bb-41ef-afd6-f905fa90bf5e', metadata={'recommendation': 245, 'calories': 450, 'name': '비빔국수'}, page_content='이름:비빔국수 \n       설명:소면을 매콤새콤한 양념에 비빈 음식'), Document(id='aa10f468-fd77-4a86-a7fa-49a18f0587b0', metadata={'calories': 450, 'name': '냉면', 'recommendation': 463}, page_content='이름:냉면 \n       설명:차가운 육수나 양념에 면을 넣은 음식')]
test end


In [18]:
## RAG에서 A 단계
def format_docs_with_metadata(docs):
    if not docs:
        return "문서가 없다. 검색된 문서가 없다"

    formatted_list= []

    for doc in docs:
        meta =doc.metadata
        info = f"""
        [메뉴정보]
        - 칼로리:{meta['calories']}
        - 이름:{meta['name']}
        - 추천수:{meta['recommendation']}
        [문서내용]
        {doc.page_content}
        """
        formatted_list.append(info)

    return "\n\n".join(formatted_list)   #join함수는 리스트에 있는 모든 값을 하나의 문자열로 합쳐라..


#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_ana = analyze_query(userquery)
query = results_ana['query']
max_cal =results_ana['max_calories']
min_rec = results_ana['min_recommendation']

#R
rdocs = retrival_docs(query, max_cal, min_rec)   # 노트북에서는 마지막 코드를 출력해서 보여줘요
#A
format_docs_with_metadata(rdocs)

매운 면 요리 {'calories': {'$lte': 1000}}
[Document(id='6890858f-24da-4add-b7d1-e89c28663a82', metadata={'name': '쫄면', 'calories': 500, 'recommendation': 226}, page_content='이름:쫄면 \n       설명:쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='c66a2cba-c5bb-41ef-afd6-f905fa90bf5e', metadata={'name': '비빔국수', 'recommendation': 245, 'calories': 450}, page_content='이름:비빔국수 \n       설명:소면을 매콤새콤한 양념에 비빈 음식'), Document(id='aa10f468-fd77-4a86-a7fa-49a18f0587b0', metadata={'recommendation': 463, 'calories': 450, 'name': '냉면'}, page_content='이름:냉면 \n       설명:차가운 육수나 양념에 면을 넣은 음식')]


'\n        [메뉴정보]\n        - 칼로리:500\n        - 이름:쫄면\n        - 추천수:226\n        [문서내용]\n        이름:쫄면 \n       설명:쫄깃한 면을 매콤한 양념에 비빈 음식\n        \n\n\n        [메뉴정보]\n        - 칼로리:450\n        - 이름:비빔국수\n        - 추천수:245\n        [문서내용]\n        이름:비빔국수 \n       설명:소면을 매콤새콤한 양념에 비빈 음식\n        \n\n\n        [메뉴정보]\n        - 칼로리:450\n        - 이름:냉면\n        - 추천수:463\n        [문서내용]\n        이름:냉면 \n       설명:차가운 육수나 양념에 면을 넣은 음식\n        '

In [21]:
# G
# RAG 최종본
def rag_process(query):
    #1번째 단계는 질의 분석 응답은 json형태
    results_ana = analyze_query(query)
    query = results_ana['query']
    max_cal =results_ana['max_calories']
    min_rec = results_ana['min_recommendation']

    #R  - 유사도 검색을 함.. 이때 메타데이터도 포함
    rdocs = retrival_docs(query, max_cal, min_rec)  
    
    #A  - [메뉴정보] [문서내용] 으로 문자열 타입으로 증강
    context = format_docs_with_metadata(rdocs)

    #프롬프트
    prompt = ChatPromptTemplate.from_template("""
    너는 음식 추천 전문가야. 아래의 참고 정보(메터데이터기반)를 바탕으로 사용자 질문에 알맞은 메뉴를 1.2. 형식으로 대답해 줘
    [사용자질문]{query}
    [참고정보 (메타데이터기반)]{context}

    [규칙]
    - 제공한 참고 정보의 메타데이터(칼로리, 추천수 등)을 답변에 활용
    - 음식 이름 먼저이야기 하기
    - 2줄 정도로 간략하게 매려적으로
    - 친절하게 구매하도록 유도
""")

    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({'query':query, 'context':context})
    return result
# 테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
rag_process(userquery)


매운 면 요리 {'calories': {'$lte': 1000}}
[Document(id='6890858f-24da-4add-b7d1-e89c28663a82', metadata={'calories': 500, 'recommendation': 226, 'name': '쫄면'}, page_content='이름:쫄면 \n       설명:쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='c66a2cba-c5bb-41ef-afd6-f905fa90bf5e', metadata={'name': '비빔국수', 'recommendation': 245, 'calories': 450}, page_content='이름:비빔국수 \n       설명:소면을 매콤새콤한 양념에 비빈 음식'), Document(id='aa10f468-fd77-4a86-a7fa-49a18f0587b0', metadata={'name': '냉면', 'recommendation': 463, 'calories': 450}, page_content='이름:냉면 \n       설명:차가운 육수나 양념에 면을 넣은 음식')]


' 지금 바로 주문해 보세요!\n\n4.  **Review against all Constraints:**\n    *   *Role assumed?* Yes.\n    *   *1. 2. format used?'

In [ ]:
## 전체 파이프라인..
#1. 질의
#2. 키워드 분석 (query, 칼로리, 추천수)
#3. 유사도분석(메타데이터 포함)
#4. 증강 ([내용정보] [문서내용])
#5. 프롬프트
#6. 문장생성